In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = SparkSession.builder \
    .appName("Gold_Layer_Hybrid_SCD_Pipeline") \
    .enableHiveSupport() \
    .getOrCreate()

In [ ]:
# 1. Read Silver Layer Parquet Data
df_cust_silver = spark.read.parquet("/silver/customers")
df_usage_silver = spark.read.parquet("/silver/usage_logs")
df_ticket_silver = spark.read.parquet("/silver/tickets")

In [ ]:
# 2. Populate Master Date Dimension
dim_date = spark.sql("""
    SELECT
        CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
        CAST(calendar_date AS DATE) AS full_date,
        day(calendar_date) AS day,
        month(calendar_date) AS month,
        quarter(calendar_date) AS quarter,
        year(calendar_date) AS year
    FROM (
        SELECT explode(sequence(to_date('2020-01-01'), to_date('2026-12-31'), interval 1 day)) AS calendar_date
    )
""")

dim_date.write.mode("overwrite").option("path", "/gold/dim_date").saveAsTable("churn_gold.DIM_DATE")

In [ ]:
# 3. HYBRID SCD1 & SCD2 MERGE FOR DIM_CUSTOMER
# ==========================================
dim_cust_table_name = "churn_gold.DIM_CUSTOMER"

incoming_cust = df_cust_silver.select(
    F.col("customer_id"),
    F.col("name"),
    F.col("email"),
    F.col("age"),
    F.col("gender"),
    F.col("geography"),
    F.col("is_active"),
    F.col("tenure_months"),
    F.to_date(F.col("date_opened")).alias("join_date"),
    F.to_timestamp(F.col("last_modified")).alias("last_modified")
)

# Version-proof check for table existence
table_exists = False
try:
    # Checkpoint immediately breaks the read lineage so we can safely overwrite later
    existing_dim = spark.table(dim_cust_table_name).localCheckpoint(eager=True)
    table_exists = True
except Exception:
    pass

if table_exists:
    max_key = existing_dim.agg(F.max("cust_key")).collect()[0][0] or 0

    active_dim = existing_dim.filter(F.col("is_current") == True)
    historical_dim = existing_dim.filter(F.col("is_current") == False)

    joined = active_dim.alias("dim").join(
        incoming_cust.alias("inc"),
        F.col("dim.customer_id") == F.col("inc.customer_id"),
        how="full_outer"
    )

    # SCD2 triggers: age, geography, is_active
    scd2_cond = (F.col("inc.age") != F.col("dim.age")) | \
                (F.col("inc.geography") != F.col("dim.geography")) | \
                (F.col("inc.is_active") != F.col("dim.is_active"))

    # SCD1 triggers: name, email, gender, tenure_months
    scd1_cond = (F.col("inc.name") != F.col("dim.name")) | \
                (F.col("inc.email") != F.col("dim.email")) | \
                (F.col("inc.gender") != F.col("dim.gender")) | \
                (F.col("inc.tenure_months") != F.col("dim.tenure_months"))

    unchanged = joined.filter(~scd2_cond & ~scd1_cond & F.col("inc.customer_id").isNotNull()).select("dim.*")

    scd1_updates = joined.filter(~scd2_cond & scd1_cond).select(
        F.col("dim.cust_key"), F.col("dim.customer_id"),
        F.col("inc.name"), F.col("inc.email"),
        F.col("dim.age"), F.col("inc.gender"), F.col("dim.geography"), F.col("dim.is_active"),
        F.col("inc.tenure_months"), F.col("dim.join_date"), F.col("dim.dw_start_date"),
        F.col("dim.dw_end_date"), F.col("dim.is_current")
    )

    expired = joined.filter(scd2_cond).select(
        F.col("dim.cust_key"), F.col("dim.customer_id"), F.col("dim.name"), F.col("dim.email"),
        F.col("dim.age"), F.col("dim.gender"), F.col("dim.geography"), F.col("dim.is_active"),
        F.col("dim.tenure_months"), F.col("dim.join_date"), F.col("dim.dw_start_date"),
        F.col("inc.last_modified").alias("dw_end_date"), F.lit(False).alias("is_current")
    )

    new_records_source = joined.filter(F.col("dim.customer_id").isNull() | scd2_cond).select("inc.*")
    insert_window = Window.orderBy("customer_id")
    new_inserts = new_records_source.withColumn("cust_key", max_key + F.row_number().over(insert_window)).select(
        "cust_key", "customer_id", "name", "email", "age", "gender", "geography", "is_active",
        "tenure_months", "join_date", F.col("last_modified").alias("dw_start_date"),
        F.to_timestamp(F.lit("9999-12-31 23:59:59")).alias("dw_end_date"), F.lit(True).alias("is_current")
    )

    missing = joined.filter(F.col("inc.customer_id").isNull()).select("dim.*")

    final_dim_customer = historical_dim.unionByName(unchanged) \
        .unionByName(scd1_updates).unionByName(expired).unionByName(new_inserts).unionByName(missing)
else:
    insert_window = Window.orderBy("customer_id")
    final_dim_customer = incoming_cust.withColumn("cust_key", F.row_number().over(insert_window)).select(
        "cust_key", "customer_id", "name", "email", "age", "gender", "geography", "is_active",
        "tenure_months", "join_date", F.col("last_modified").alias("dw_start_date"),
        F.to_timestamp(F.lit("9999-12-31 23:59:59")).alias("dw_end_date"), F.lit(True).alias("is_current")
    )
    final_dim_customer = final_dim_customer.localCheckpoint(eager=True)

final_dim_customer.write.mode("overwrite").option("path", "/gold/dim_customer").saveAsTable(dim_cust_table_name)